# 5. Analysis and Plots

This notebook finds the latest completed baseline and AttnRes runs, regenerates the comparison plots, and displays the resulting images.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/AtinChing/AttnResGPT-mini.git'
REPO_NAME = 'AttnResGPT-mini'

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

candidates = [Path(f'/content/{REPO_NAME}'), Path(f'/content/drive/MyDrive/{REPO_NAME}'), Path.cwd()]
repo_root = next((p for p in candidates if (p / 'requirements.txt').exists() and (p / 'src').exists()), None)

if repo_root is None:
    target = Path(f'/content/{REPO_NAME}')
    print(f'Cloning {REPO_URL} into {target} ...')
    subprocess.run(['git', 'clone', REPO_URL, str(target)], check=True)
    repo_root = target
else:
    print(f'Using existing repo at {repo_root}')

%cd {repo_root}
!pip -q install -r requirements.txt

In [ ]:
import torch
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

baseline_candidates = sorted(Path('runs').glob('*baseline*'))
attnres_candidates = sorted(Path('runs').glob('*attnres*'))
if not baseline_candidates or not attnres_candidates:
    raise FileNotFoundError('Run the training notebooks first so there are saved run directories under runs/.')
baseline_run = baseline_candidates[-1]
attnres_run = attnres_candidates[-1]
print('baseline_run:', baseline_run)
print('attnres_run:', attnres_run)
!python scripts/compare_runs.py --baseline-run {baseline_run} --attnres-run {attnres_run}
!python scripts/plot_metrics.py --run-dirs {baseline_run} {attnres_run} --output-dir plots/nb5_analysis

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for image_path in sorted(Path('plots/nb5_analysis').glob('*.png')):
    print(image_path.name)
    display(Image(filename=str(image_path)))